# 09 — Elyra：GRIB preprocess（CPU）

呼叫映像內 `/app/run-preprocess.sh`（與 preprocess Job 相同）。

**Runtime Image 必須是** `quay.io/cwa/rhoai/corrdiff-preprocess:latest`（含 eccodes／pygrib），**不要**用 Data Science／CorrDiff GPU 映像。

| 環境變數 | smoke 預設（notebook 內 setdefault） | 說明 |
|----------|--------------------------------------|------|
| `DATE` | `20260707` | |
| `SHOME` | `/mnt/corrdiff` | |
| `SKIP_PREPROCESS` | `0` | `1` = 強制跳過 |
| `SKIP_SIZE_CHECKS` | `1` | 三日 GRIB 必開；完整 47 檔設節點 env=`0` |
| `NUM_LEADDAY` / `OP_NUM_LEADDAY` | `2` / `1` | 三日 smoke |
| `FORCE_CONFIG` | `1` | 重產 config 並套 lead patch |

`INPUT_MODE` 為 `existing`／`url`（由 `08` 寫入 marker）時自動 skip。

> 注意：Elyra 節點 env 不一定進 kernel；smoke 預設寫在本 notebook，勿只依賴 Properties。


In [ ]:
import os
import subprocess
from pathlib import Path

SHOME = os.environ.get("SHOME", "/mnt/corrdiff")
DATE = os.environ.get("DATE", "20260707")
SKIP = os.environ.get("SKIP_PREPROCESS", "0").strip() == "1"

workdir = Path(SHOME) / "workdir" / DATE
marker = workdir / ".elyra_input_mode"
input_nc = workdir / f"CorrdiffInput_EC_RAW_{DATE}.nc"
script = Path("/app/run-preprocess.sh")

mode = marker.read_text().strip() if marker.is_file() else os.environ.get("INPUT_MODE", "grib")
print(f"DATE={DATE} marker_mode={mode} SKIP_PREPROCESS={SKIP}")

if SKIP or mode in ("existing", "url"):
    assert input_nc.is_file(), f"Skip requested but missing {input_nc}"
    print(f"[SKIP] preprocess (mode={mode}); input OK ({input_nc.stat().st_size} bytes)")
    print("preprocess completed successfully (skipped).")
else:
    assert script.is_file(), (
        f"Missing {script}. Set Elyra Runtime Image to "
        "quay.io/cwa/rhoai/corrdiff-preprocess:latest (not Data Science)."
    )
    subprocess.run(
        ["python3", "-c", "import eccodes, pygrib, netCDF4; print('deps OK')"],
        check=True,
    )
    # Elyra node env_vars do not always reach the kernel; default to 3-day smoke.
    # Full 47-file run: set node env SKIP_SIZE_CHECKS=0 (overrides setdefault).
    env = os.environ.copy()
    env.setdefault("SHOME", SHOME)
    env.setdefault("HOME", "/tmp")
    env.setdefault("SKIP_SIZE_CHECKS", "1")
    env.setdefault("NUM_LEADDAY", "2")
    env.setdefault("OP_NUM_LEADDAY", "1")
    env.setdefault("FORCE_CONFIG", "1")
    print(
        "preprocess env:",
        f"SKIP_SIZE_CHECKS={env.get('SKIP_SIZE_CHECKS')}",
        f"NUM_LEADDAY={env.get('NUM_LEADDAY')}",
        f"OP_NUM_LEADDAY={env.get('OP_NUM_LEADDAY')}",
        f"FORCE_CONFIG={env.get('FORCE_CONFIG')}",
    )
    print("Running:", script, DATE)
    proc = subprocess.run(
        ["bash", str(script), DATE],
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise AssertionError(
            f"run-preprocess.sh failed (exit {proc.returncode}). "
            "3-day GRIB needs SKIP_SIZE_CHECKS=1 (notebook defaults this)."
        )
    assert input_nc.is_file(), f"Missing output {input_nc}"
    print(f"[OK] {input_nc} ({input_nc.stat().st_size} bytes)")
    print("preprocess completed successfully.")
